# Symbolic Regression on Feynman Equations using Genetic Programming

Symbolic regression is a type of regression analysis where the goal is to discover mathematical expressions that best describe a given dataset. Unlike traditional regression, symbolic regression does not assume a predefined model structure. Instead, it searches for both the structure and the parameters that best fit the data. This approach can yield interpretable, analytical models that help uncover underlying relationships in the data.

In this notebook, we will apply symbolic regression to a set of well-known physical equations: the Feynman Equations. These equations, derived by physicist Richard Feynman, describe fundamental physical phenomena in areas such as mechanics, electromagnetism, and thermodynamics.

We will perform symbolic regression with **genetic programming** (GP), using `gplearn`, a `scikit-learn`-inspired Python library for GP.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import  gplearn.genetic as gp
import matplotlib.pyplot as plt
import graphviz
import pickle

Read the metadata of the equations from the csv file plus the equations from the pickle file.

In [ ]:
eq_df = pd.read_csv('data/FeynmanEquations.csv')
eq_df = eq_df.dropna(axis = 0, how = 'all')
eq_df.head()

In [ ]:
datasets = {}
with open("./data/feynman_eq_3_8.pkl", "rb") as f:
    datasets = pickle.load(f)

Now, for each equation in the dataset, perform symbolic regression using GP. Split each dataset into a training set and a validation set. Select a validation metric to evaluate performance, noting that this metric is not necessarily the same as the GP fitness function. Experiment with different sets of hyperparameters to observe how the results change.

Take a look at the [documentation](https://gplearn.readthedocs.io/en/stable/intro.html).

In [ ]:
from gplearn.genetic import BaseSymbolic
from sklearn.utils.validation import validate_data

BaseSymbolic._validate_data = lambda self, *args, **kwargs: validate_data(
    self,
    *args,
    **kwargs,
)

Define some function with `make_function`.

In [ ]:
from gplearn.functions import make_function

# TODO
def squ(x):
    return 1 / (1+np.e**x)
make_function(function=squ, name='squ', arity=1)

Set the hyperparameters.

In [ ]:
random_state = 0

#hyperparameters
max_gen = 50
fset = ('add', 'sub', 'mul', 'div', 'sqrt', 'inv', 'neg', 'abs', 'sin', 'cos') # add the function that you implemented
pop_size = 1500
tournament_size = 20
parsimony_coefficient = 0.015
p_crossover=0.9
p_subtree_mutation = 0.03
p_hoist_mutation=0.01
p_point_mutation=0.01
fitness = 'rmse'
val_fit = r2_score

And now create the optimization loop.

Iterate over each dataset in the dictionary, collect the results (as described by the definition of `results`) and collect the validation fitness history (to later print them).

During each cycle take X and y from the dataset (X = the entire dataset without the target), y is the target. Remember to convert them to numpy ndarray.

Use train_test_split in order to divide into training and test set.

Use the name of the columns (without the target one) as feature names.

Instantiate a `gp.SymbolicRegressor` with all the hyperparameters set before (set `generations` to 1 and `warm_start=True`) and then loop over the generations. Each generation must:

1. set the parameters
2. fit to X and y (train set)
3. compute the validation error (test set)
4. store the value to later plot

Once the generation is ended, store the final results and go on.

In [ ]:
np.random.seed(random_state)
results = pd.DataFrame(columns=['dataset','best_fit', 'original', 'equation'])
val_histories = {}

for name, df in datasets.items():
    X = df.drop(columns=['target'])
    y = df['target']
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=random_state)
    
    val_history = []
    var_names = df.drop(columns=['target']).columns.tolist()
    
    sr = gp.SymbolicRegressor(population_size=pop_size, tournament_size=tournament_size, generations=1, warm_start=True, parsimony_coefficient=parsimony_coefficient, p_hoist_mutation=p_hoist_mutation, p_subtree_mutation=p_subtree_mutation, p_point_mutation=p_point_mutation, p_crossover=p_crossover, function_set=fset, metric=fitness, verbose=0, )

    for i in range(0, max_gen+1):
        # TODO 
        # set parameters, fit, predict, append validation fit to the val_history
        sr.set_params(generations=i+1)
        sr.fit(X_train, y_train)

        predictions = sr.predict(X_val)
        val_score = val_fit(y_val, predictions)
        val_history.append(val_score)

    correct_name = str(name).removeprefix("feynman_").replace("_", ".")
    orig = eq_df.loc[eq_df["Filename"].eq(correct_name), "Formula"].tolist()

    best_fit = val_history[-1]
    val_histories[name] = val_history
    results.loc[len(results)] = [name,
                                 best_fit,
                                 orig[0] if orig else None,
                                 sr._program,
                                 ]
    print(f'{name} best_fit: {best_fit}')

print("------------------------------------")
print(results)

Plot the loss histories!

In [ ]:
for name, history in val_histories.items():
    # TODO
    # plot the history
    plt.plot(history)
    plt.title(name)
    plt.show()

You can also try this simple tool to visualize a bit the solution found.

In [ ]:
dot_data=results.loc[3, "equation"].export_graphviz()
graphviz.Source(dot_data)